# 03 — Análise de risco em texto (NLP) — 100% offline

Demonstra a fatia de **texto** rodando sem nuvem: léxico de risco + classificador
supervisionado (TF-IDF + Naive Bayes) + sentimento + fusão em nível de alerta.

Usa os relatos **sintéticos** de `data/samples/`. **Sem PHI.**

In [ ]:
# Garante que a raiz do projeto esteja importavel
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from backend.app.ports.nlp import LocalNlpAdapter
from backend.app.services.text.nlp import analisar_texto

nlp = LocalNlpAdapter()
print("Pipeline de texto carregada (backend local).")

In [ ]:
# Roda a analise sobre cada amostra sintetica
samples = sorted(Path("../data/samples").glob("*.txt"))
for arq in samples:
    texto = arq.read_text(encoding="utf-8")
    r = analisar_texto(texto, nlp)
    print("=" * 70)
    print("Amostra:", arq.name)
    print("Nivel de alerta:", r.nivel_alerta.value)
    print("Sentimento:", r.sentimento.rotulo, f"({r.sentimento.score})")
    for c in r.categorias_risco:
        print(f"  risco -> {c.categoria} (score={c.score}) evidencias={c.evidencias}")
    print("Acao recomendada:", r.acao_recomendada)

In [ ]:
# Inspecionar o classificador supervisionado isoladamente
from backend.app.services.text.classifier import prever_categoria

frases = [
    "sinto o coracao acelerado e um aperto no peito",
    "ele me empurrou e tenho medo em casa",
    "estou tranquila e feliz com a gravidez",
]
for f in frases:
    cat, prob = prever_categoria(f)
    print(f"{f!r} -> {cat} (prob={prob:.2f})")